In [ ]:
# ✅ Scalable File Upload Design (Python + S3 + SQS + SNS)
🏗 High-Level Architecture

Client
   |
   v
API Service (FastAPI)
   |
   v
S3 (store file)
   |
   v
SQS (processing queue)
   |
   v
Worker Service (Python consumers)
   |
   v
SNS (notify other services / email / webhook)


In [ ]:
1️⃣ S3
    Stores large files
    Highly scalable
    Cheap storage


In [ ]:
from fastapi import FastAPI, File, UploadFile
import boto3
import uuid
import os
import json

app = FastAPI()

S3_BUCKET = "your-bucket-name"
SQS_QUEUE_URL = "your-sqs-url"

s3 = boto3.client("s3")
sqs = boto3.client("sqs")

@app.post("/upload")
async def upload_file(file: UploadFile = File(...)):
    file_id = str(uuid.uuid4())
    file_key = f"uploads/{file_id}_{file.filename}"

    # Upload to S3
    s3.upload_fileobj(file.file, S3_BUCKET, file_key)

    # Send message to SQS
    message = {
        "file_id": file_id,
        "file_key": file_key,
        "bucket": S3_BUCKET
    }

    sqs.send_message(
        QueueUrl=SQS_QUEUE_URL,
        MessageBody=json.dumps(message)
    )

    return {"status": "uploaded", "file_id": file_id}


In [ ]:
| SQS                    | SNS                      |
| ---------------------- | ------------------------ |
| Message queue          | Pub/Sub system           |
| Pull-based             | Push-based               |
| 1 consumer per message | Multiple subscribers     |
| Stores messages        | Does not store long-term |


In [ ]:
4️⃣ What are the types of SQS?

Answer:

Standard Queue
FIFO Queue

In [ ]:
| Standard               | FIFO             |
| ---------------------- | ---------------- |
| At-least-once delivery | Exactly-once     |
| Best-effort ordering   | Strict ordering  |
| High throughput        | Lower throughput |


In [ ]:
1️⃣4️⃣ How would you handle 10 million messages per hour?

Answer:
    Use Standard Queue
    Batch processing (10 messages per call)
    Horizontal scaling (Auto Scaling Group / ECS)
    Long polling
    Proper visibility timeout

In [ ]:
| Short Polling        | Long Polling           |
| -------------------- | ---------------------- |
| Returns immediately  | Waits up to 20 seconds |
| More empty responses | Fewer empty responses  |
| Higher cost          | Lower cost             |
| Less efficient       | More efficient         |

    

In [ ]:
SNS is used for event broadcasting using pub/sub model.
SQS is used for reliable asynchronous processing.
Together they provide decoupling, scalability, fault tolerance, and eventual consistency in distributed systems.

In [ ]:
Q: Your application traffic suddenly increased 10x and users are seeing latency. What will you do?

Check CloudWatch metrics (CPU, memory, RDS connections)
Enable Auto Scaling for EC2/ECS
Use ALB
Add Redis (ElastiCache) for caching
Enable CloudFront CDN
Scale RDS vertically or add read replicas

In [ ]:
Q: Database CPU is constantly 100%. How do you fix it?

    Check slow query logs
    Add proper indexing
    Use read replicas
    Move read-heavy traffic to cache
    Consider Aurora if scaling required

In [ ]:
🔥 7️⃣ Large File Uploads Causing API Timeout

Use S3 pre-signed URLs
Client uploads directly to S3
API only generates URL

In [ ]:
🔥 8️⃣ 1 Million Messages Per Hour Processing

Use SQS Standard
Batch receive (10 messages)
Horizontal scaling
Long polling
Proper visibility timeout

In [ ]:
🔥 9️⃣ How To Secure S3 Bucket?

Block public access
IAM role-based access
Enable bucket policy
Enable encryption (SSE-S3 / KMS)
Enable access logging

In [ ]:
🔥 1️⃣2️⃣ Lambda Timeout Issue

Answer:

Increase memory (more CPU)
Increase timeout limit
Move heavy job to ECS/Batch
Use async processing

In [ ]:
🔥 1️⃣5️⃣ Disaster Recovery Strategy

Multi-AZ for high availability
Cross-region replication
RDS snapshots
S3 versioning + replication
Backup strategy


In [ ]:
🔥 1️⃣6️⃣ How To Handle Sudden DB Connection Exhaustion?

    Use connection pooling
    Increase DB max connections
    Use RDS Proxy
    Reduce idle connections

In [ ]:
🔥 1️⃣7️⃣ Cost Suddenly Increased
   . Use S3 lifecycle policies
   . Use Savings Plans
   . Look for unused EC2


In [ ]:
🔥 2️⃣0️⃣ How To Make System Highly Available?

Multi-AZ architecture
Load balancer
Auto Scaling
Stateless services
DB replication
Health checks

In [ ]:
❓ How do you design a system that never loses data?

SQS for reliable messaging
DLQ for failures
Multi-AZ RDS
S3 versioning
Backups
Monitoring & alerting

In [ ]:
My architecture is horizontally scalable, decoupled using SQS/SNS, 
stateless services behind ALB, multi-AZ RDS, Redis caching, and
full observability using CloudWatch and X-Ray.
    It is designed for high availability, fault tolerance, and cost optimization.

In [ ]:
1️⃣ How do you optimize a slow Python API?

Add DB indexes
Reduce N+1 queries
Use caching (Redis)
Use connection pooling
Profile using cProfile
Use async if I/O heavy

In [ ]:
3️⃣ How do you handle transactions across microservices?

Use Saga pattern
Orchestration via Step Functions
Make APIs idempotent
Use compensating transactions

In [ ]:
6️⃣ How do you scale Django?

Run behind Gunicorn + Nginx
Use Redis cache
Use RDS read replicas
Deploy using Kubernetes
Enable autoscaling

In [ ]:
7️⃣ How do you secure Django app?

Use HTTPS
CSRF protection
Use JWT or OAuth
Validate inputs
Store secrets in AWS Secrets Manager

In [ ]:
8️⃣ How do you optimize slow MySQL query?

Answer:
    Add proper indexes
    Avoid SELECT *
    Use EXPLAIN
    Normalize schema
    Avoid unnecessary joins

In [ ]:
9️⃣ What is indexing?

Index improves search speed but increases write cost.

In [ ]:
🔟 How do you handle DB connection exhaustion?

    Use connection pooling
    Increase max_connections
    Close connections properly
    Use read replicas
    Add caching layer

In [ ]:
EC2 → manual server management
ECS → managed container service
EKS → Kubernetes managed by AWS

In [ ]:
1️⃣2️⃣ How do you design highly available system?

    Multi-AZ RDS
    ALB
    Auto Scaling
    Use S3 for storage
    Health checks

In [ ]:
FIFO → ordered + no duplicates
Standard → high throughput, best-effort ordering

In [ ]:
1️⃣5️⃣ How to secure AWS?

    IAM roles
    Least privilege
    VPC
    Security groups
    WAF
    CloudTrail monitoring

In [ ]:
❓ What is GIL? How do you handle concurrency?

GIL (Global Interpreter Lock) allows only one thread to execute Python bytecode at a time.
For CPU-bound tasks → I use multiprocessing.
For I/O-bound tasks → I use asyncio, threading, or async frameworks like FastAPI.

In [ ]:
❓ How do you optimize Python performance?

Use profiling tools (cProfile, line_profiler)
Avoid unnecessary loops
Use generators instead of lists
Use caching (LRU cache / Redis)
Use async for I/O heavy systems
Optimize DB queries instead of Python code first

In [ ]:
Multithreading → shared memory, affected by GIL, best for I/O

Multiprocessing → separate memory, true parallelism, best for CPU tasks

In [ ]:
❓ What is monkey patching?

Dynamically modifying a class or module at runtime.

In [ ]:
❓ How does Django ORM prevent SQL injection?

It uses parameterized queries internally instead of string concatenation.

In [ ]:
❓ How do you handle authentication in FastAPI?

Using OAuth2 + JWT tokens.
Access token validation using dependency injection.

In [ ]:
❓ How do you scale FastAPI?

   Run with Gunicorn + Uvicorn workers
   Horizontal scaling using Kubernetes
   Add Redis caching
   Use load balancer (ALB / Nginx)

In [ ]:
SNS → Pub/Sub (fan-out architecture)
SQS → Queue-based, decouples services
SNS + SQS used for event-driven architecture

In [ ]:
❓ How do you handle sudden DB connection exhaustion?
    Use connection pooling
    Use RDS Proxy
    Reduce idle connections
    Implement retry with exponential backoff


In [ ]:
EC2 → Virtual machine
ECS → AWS container service
EKS → Managed Kubernetes

In [ ]:
❓ How to design file upload scalable system?

Client
   |
   v
API Gateway / Load Balancer
   |
   v
Upload Service (Stateless)
   |
   v
Object Storage (S3 / GCS / Blob)
   |
   +--> Message Queue (SQS/Kafka)
           |
           v
     Processing Service
       (Virus scan, thumbnail, metadata)
           |
           v
        Database (MySQL / DynamoDB)
1️⃣ Basic Scalable Approach (IMPORTANT)
   Upload directly to object storage (S3)
2️⃣ Use Pre-Signed URL (Industry Standard)
    .  Client asks backend:
            POST /generate-upload-url
    .   Backend generates S3 Pre-Signed URL
    .   Client uploads directly to S3
    .   Backend stores metadata in DB
3️⃣ Handling Large Files (Multipart Upload)
         Use S3 Multipart Upload
File split into chunks (5MB+)
    Upload parts in parallel
    Retry only failed chunks
    Complete upload API call

Benefits:
    Faster
    Resume support
    Fault tolerant
5️⃣ Asynchronous Processing
S3 → Event → SQS

Worker consumes message

In [ ]:
❓ Difference between CMD and ENTRYPOINT?

CMD provides a default command or arguments that can be overridden when running the container.

ENTRYPOINT defines the main executable of the container and runs every time,
with CMD acting as its default arguments.

In [ ]:
❓ How do you reduce Docker image size?
. Use slim/alpine base images
. Multi-stage builds
. Remove unnecessary packages
. Use .dockerignore

In [ ]:
Deployment → stateless apps
StatefulSet → stateful apps with stable network identity

In [ ]:
❓ How do you handle zero downtime deployment?

Rolling updates with readiness & liveness probes configured properly.

In [ ]:
Index improves read performance but increases write cost.

In [ ]:
Normalization → remove redundancy

Denormalization → improve read performance

In [ ]:
❓ How did you handle transactions across multiple services?

Used Saga pattern.
Either orchestration via Step Functions
choreography via event

In [ ]:
❓ How do you design high traffic backend?

Load balancer
Stateless app servers
Redis caching
DB read replicas
Async processing with SQS
Auto-scaling

In [ ]:
❓ How do you make API idempotent?
Unique request ID
DB unique constraint
Store request status

In [ ]:
❓ How do you prevent duplicate payment?

Idempotency key
DB constraint
Distributed lock (Redis)

In [ ]:
❓ How do you handle rate limiting?

API Gateway rate limit
Redis-based token bucket algorithm

In [ ]:
❓ If your service CPU is 90% and response time increased, what will you check?

Answer:

Check logs
Check DB slow queries
Check memory leak
Check thread blocking
Check recent deployment changes
Check autoscaling metrics

In [ ]:
An Idempotency Key is a unique identifier sent with a request to ensure the same request is processed only once,

2️⃣ How do you avoid duplicate processing in SQS?
   . Use idempotency key
   . Use FIFO queue with deduplication ID
   . Maintain processed message table

In [ ]:
Standard → At-least-once, unordered
FIFO → Exactly-once (with dedup), ordered per MessageGroupId

In [ ]:
4️⃣ How does SNS fan-out work?

SNS publishes message → multiple subscribers (SQS, Lambda, HTTP endpoints) receive it simultaneously.

In [ ]:
5️⃣ Lambda cold start — how do you reduce?

Use provisioned concurrency
Reduce package size
Keep function warm
Use smaller runtime

In [ ]:
ALB → Layer 7 (HTTP/HTTPS routing)
NLB → Layer 4 (TCP/UDP, high performance)

In [ ]:
7️⃣ How do you secure S3 bucket?

Block public access
IAM policy
Bucket policy
Enable encryption (SSE-S3 or SSE-KMS)

In [ ]:
8️⃣ How do you implement zero downtime deployment?

Blue-Green deployment
Rolling update
Use Auto Scaling + Load balancer

In [ ]:
9️⃣ How does Auto Scaling work?

Monitors CloudWatch metrics → adds/removes EC2 instances based on thresholds.

In [ ]:
🔟 What is RDS Proxy? Manages DB connections

In [ ]:
Multi-AZ → high availability (failover)
Read replica → scale reads

In [ ]:
IAM user → person
IAM role → temporary access for services

In [ ]:
1️⃣8️⃣ What is Dead Letter Queue?

Queue where failed messages are stored for debugging.

In [ ]:
2️⃣0️⃣ How do you store secrets securely?

AWS Secrets Manager

Parameter Store

IAM roles (never hardcode)

In [ ]:
2️⃣1️⃣ How do you design event-driven architecture in AWS?

SNS for publish
SQS for decoupling
Lambda for processing
DLQ for failures


In [ ]:
CDN service to cache static content globally.

In [ ]:
EBS → attached to single EC2
EFS → shared file system

In [ ]:
2️⃣4️⃣ How do you secure microservices in AWS?

VPC
Security Groups
IAM roles
API Gateway auth
WAF

In [ ]:
2️⃣6️⃣ How do you reduce AWS cost?

Right size instances
Use Spot instances
Reserved instances
S3 lifecycle policies
Auto scaling

In [ ]:
Public → has route to Internet Gateway
Private → no direct internet access

In [ ]:
2️⃣9️⃣ How do you debug Lambda in production?

CloudWatch logs
X-Ray tracing
Add structured logging
Monitor DLQ

In [ ]:
❓ Your API latency suddenly increased from 200ms to 3 seconds. What will you check?

Check CloudWatch / Prometheus metrics
Check CPU, memory
Check DB slow queries
Check thread blocking
Check downstream service latency
Check recent deployment changes
Check autoscaling health

In [ ]:
❓ How did you handle transactions across multiple services?

   We used Saga pattern.
       . For simple flows  Choreography using SNS/SQS events
       . For complex workflows → Orchestration using AWS Step Functions
       . Used idempotency keys
    

In [ ]:
✅ What is RDS Proxy?

RDS Proxy is a fully managed database proxy for Amazon RDS
that helps manage database connections efficiently.

In [ ]:
❓ Suddenly database connections reached max. What will you do?
   . Restart stuck pods
   . Scale read replicas
   . Enable RDS Proxy